In [10]:
%load_ext autoreload
%autoreload 2

import sys, os

import pandas as pd
from pathlib import Path
from src.inference import ASRInference
from src.metrics import cer


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
CKPT = Path("logs/unsplittable-v2/version_0/checkpoints/best_epoch17_harmonic_3.24.ckpt")
asr = ASRInference(CKPT, device="cuda")
print("Model loaded.")


Model loaded.


/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(


In [12]:
dev_df = pd.read_csv("data/dev.csv")
paths = [Path("data") / fn for fn in dev_df["filename"]]

preds = asr.transcribe(paths)

errors = [cer(p, str(t)) for p, t in zip(preds, dev_df["transcription"])]
print(f"Mean CER on dev: {sum(errors)/len(errors):.4f}")
for p, t in list(zip(preds, dev_df["transcription"]))[:5]:
    print(f"  pred={p!r:10s}  truth={t}")


Mean CER on dev: 0.0412
  pred='849905'    truth=849905
  pred='967653'    truth=967653
  pred='427524'    truth=427524
  pred='93752'     truth=93752
  pred='43355'     truth=43355


In [8]:
from pathlib import Path
import pandas as pd

print("test.csv:", Path("data/test.csv").exists())
print("sample_submission.csv:", Path("data/sample_submission.csv").exists())
print("test/ files:", len(list(Path("data/test").glob("*"))))

test_df = pd.read_csv("data/test.csv")
print(f"\ntest.csv: {len(test_df)} rows")
print(test_df.head())
print("\ncolumns:", list(test_df.columns))


test.csv: True
sample_submission.csv: True
test/ files: 2582

test.csv: 2582 rows
              filename  ext  samplerate
0  test/d2440788a9.mp3  mp3       16000
1  test/e247dbf761.mp3  mp3       16000
2  test/071f4a5be7.mp3  mp3       16000
3  test/798bd15db7.mp3  mp3       16000
4  test/58c0464ad5.mp3  mp3       16000

columns: ['filename', 'ext', 'samplerate']


In [9]:
first_path = Path("data") / test_df.iloc[0]["filename"]
print(f"First file: {first_path}  exists={first_path.exists()}")
pred = asr.transcribe([first_path])
print(f"Prediction: {pred[0]}")


First file: data/test/d2440788a9.mp3  exists=True
Prediction: 461694


In [17]:
paths = [Path("data") / fn for fn in test_df["filename"]]
preds = asr.transcribe(paths)   # ~3–5 min for ~1000 files

submission = pd.DataFrame({
    "filename":      test_df["filename"],
    "transcription": preds,
})
submission.to_csv("submission.csv", index=False)
print(f"Wrote {len(submission)} rows")
print(submission.head())


Wrote 2582 rows
              filename transcription
0  test/d2440788a9.mp3        461694
1  test/e247dbf761.mp3        207723
2  test/071f4a5be7.mp3         79187
3  test/798bd15db7.mp3         64048
4  test/58c0464ad5.mp3        861168
